# Volume 3, Chapter 4: Inequalities, Large Numbers and Entropy (Python companion)

Exercises first, solutions at the end. We put Chebyshev, Hoeffding and Chernoff side by side against exact binomial tails, watch Bernstein polynomials converge, and then turn to computer science: the maximum load of balls in bins, Bloom filters, the median trick for boosting a randomised algorithm, Huffman codes against the entropy, typical sequences, and the longest run of heads.

*Before each solution, predict what the output will look like.*

In [ ]:
import math, heapq, hashlib, zlib
from math import comb, log, log2
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(4)

## Exercises

**P4.1 (★) Three bounds against the truth.** For $1000$ fair coin tosses, compute exactly $P(S\ge600)$, and compare with Chebyshev, the simplified Chernoff bound $e^{-\delta^2\mu/3}$ (Theorem 4.6), Hoeffding (Theorem 4.9) and the two bounds of Theorem 4.7. Then redraw the figure of the chapter for $p=0.1$ and frequency $0.2$.

**P4.2 (★) Bernstein polynomials.** Compute $\sup_x|B_nf(x)-f(x)|$ for $f(x)=|x-\frac12|$ and $f(x)=\sin(2\pi x)$, for $n=4,16,\dots,4096$. Find the rate in each case: is the error of order $\frac1{\sqrt n}$ or $\frac1n$? (Exercise 4.2 gives an upper bound for the first.)

**P4.3 (★★) Balls into bins.** Throw $n$ balls into $n$ bins for $n=10^3,\dots,10^7$ and record the maximum load. Compare with $\frac{\ln n}{\ln\ln n}$ and with the bound $\lceil\frac{3\ln n}{\ln\ln n}\rceil$ of Exercise 4.5. Then try the *power of two choices*: each ball looks at two random bins and goes to the less loaded one.

**P4.4 (★★) A Bloom filter.** Implement a Bloom filter with $m=10n$ bits for $n=20\,000$ keys, with $k$ hash positions derived from SHA-256. Measure the false positive rate on $200\,000$ other keys for $k=1,\dots,14$, and compare with $(1-e^{-kn/m})^k$ and with the optimum of Exercise 4.6.

**P4.5 (★★) The median trick.** Estimate $\pi$ by throwing $400$ random points in the unit square and counting those in the quarter disc. Find $\varepsilon$ such that one estimate is within $\varepsilon$ of $\pi$ about $75\%$ of the time. Then take the median of $2k+1$ independent estimates and compare its failure rate with the bound $e^{-(2k+1)/8}$ of Exercise 4.7. Compare with the *mean* of $2k+1$ estimates: which is better, and why does the median trick still matter?

**P4.6 (★★) Huffman against the entropy.** (a) Implement Huffman's algorithm. For independent Bernoulli$(0.1)$ bits, code blocks of $n=1,\dots,10$ bits and check $nH\le L_n<nH+1$ (Theorem 4.19). (b) For a paragraph of English, compute the entropy of the letter frequencies and the length of a Huffman code for letters, then for pairs of letters, and compare with `zlib`.

**P4.7 (★) Typical sequences.** For Bernoulli$(0.2)$ bits and $\varepsilon=0.05$, compute exactly, for $n=100,500,2000,10\,000$, the probability of the typical set $T_\varepsilon^{(n)}$ and $\frac1n\log_2|T_\varepsilon^{(n)}|$, and check the bounds of Theorem 4.21.

**P4.8 (★★) Runs of heads.** (a) Estimate the law of the longest run of heads in $200$ tosses. (b) A person faking coin tosses tends to alternate: model this by a sequence that switches with probability $0.6$ at each step. How long are its longest runs? (c) Follow $\frac{L_n}{\log_2n}$ along one sequence of $10^7$ tosses (Exercise 4.10).

## Solutions

**P4.1.** The exponent of each bound is what matters; we print natural logarithms to compare them.

In [ ]:
n, a = 1000, 600
exact = sum(comb(n, k) for k in range(a, n + 1)) / 2**n
mu, var = n / 2, n / 4
I = lambda p, x: x * log(x / p) + (1 - x) * log((1 - x) / (1 - p))
bounds = {
    'Chebyshev  Var/t^2': var / (a - mu)**2,
    'Chernoff   e^{-d^2 mu/3}': math.exp(-(0.2**2) * mu / 3),
    'Hoeffding  e^{-2t^2/n}': math.exp(-2 * (a - mu)**2 / n),
    'Chernoff   e^{-n I(0.6)}': math.exp(-n * I(0.5, 0.6)),
    'lower      e^{-n I}/(n+1)': math.exp(-n * I(0.5, 0.6)) / (n + 1),
}
print(f'exact: {exact:.3e}')
for name, b in bounds.items():
    print(f'{name:28s} {b:.3e}   ratio to exact {b/exact:.3g}')

Chebyshev is off by a factor of $10^8$, the simplified Chernoff bound by $10^7$; Hoeffding and the optimal Chernoff bound are within a factor $15$, and the lower bound of Theorem 4.7 is about $75$ times too small. The truth is about $\frac{c}{\sqrt n}e^{-nI}$: the CLT of Chapter 6 refined by Stirling explains the missing $\sqrt n$.

In [ ]:
p, x = 0.1, 0.2
ns = np.arange(10, 301, 10)
def tail(n, p, x):
    k0 = math.ceil(x * n - 1e-12)
    return sum(comb(n, k) * p**k * (1 - p)**(n - k) for k in range(k0, n + 1))
ex = [tail(n, p, x) for n in ns]
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.semilogy(ns, ex, 'k.', label='exact')
ax.semilogy(ns, np.minimum(1, 9 / ns), label='Chebyshev $9/n$')
ax.semilogy(ns, np.exp(-2 * ns * (x - p)**2), label='Hoeffding')
ax.semilogy(ns, np.exp(-ns * I(p, x)), label='Chernoff $e^{-nI}$')
ax.set_xlabel('n'); ax.legend(); plt.show()
slope = np.polyfit(ns[10:], np.log(ex[10:]), 1)[0]
print('fitted slope of log P:', slope, '  I_p(x) =', I(p, x))

The fitted slope approaches $-I_{0.1}(0.2)=-0.0444$ slowly, because of the polynomial correction $-\frac12\ln n$ that the exponential rate ignores.

**P4.2.** $B_nf(x)=\mathbb E f(S_n/n)$; we evaluate it on a grid with the binomial probabilities computed by a stable recursion in log space.

In [ ]:
def bernstein(f, n, xs):
    k = np.arange(n + 1)
    logc = np.array([math.lgamma(n + 1) - math.lgamma(j + 1) - math.lgamma(n - j + 1) for j in k])
    out = []
    for x in xs:
        if x in (0.0, 1.0):
            out.append(f(x)); continue
        w = np.exp(logc + k * math.log(x) + (n - k) * math.log(1 - x))
        out.append(np.dot(w, f(k / n)))
    return np.array(out)
xs = np.linspace(0, 1, 801)
fs = {'|x-1/2|': lambda t: np.abs(t - 0.5), 'sin(2 pi x)': lambda t: np.sin(2 * np.pi * t)}
for name, f in fs.items():
    print(name)
    for n in [4, 16, 64, 256, 1024, 4096]:
        err = np.max(np.abs(bernstein(f, n, xs) - f(xs)))
        print(f'   n = {n:5d}: sup error {err:.5f},  sqrt(n) * err = {math.sqrt(n)*err:.4f},  n * err = {n*err:.4f}')

For the kink, $\sqrt n\cdot$error tends to $\frac1{\sqrt{2\pi}}\approx0.399$: at $x=\frac12$ the error is $\mathbb E|\frac{S_n}n-\frac12|$, and the CLT of Chapter 6 gives $\frac1{2\sqrt n}\mathbb E|Z|=\frac1{2\sqrt n}\sqrt{\frac2\pi}$. For the smooth sine, $n\cdot$error tends to $\max_x\frac{x(1-x)}2|f''(x)|=2\pi^2\max_xx(1-x)|\sin2\pi x|\approx2\pi^2\times0.1998\approx3.94$: a Taylor expansion to second order gives $B_nf(x)-f(x)\approx\frac{x(1-x)}{2n}f''(x)$. Bernstein polynomials are slow, which is why they are used for their shape-preserving properties (Bézier curves) rather than for numerical approximation.

**P4.3.** `np.bincount` counts the balls per bin in one call.

In [ ]:
for e in range(3, 8):
    n = 10**e
    loads = [int(np.bincount(rng.integers(0, n, n), minlength=n).max()) for _ in range(5)]
    L = math.log(n)
    print(f'n = 10^{e}: max loads {loads},  ln n/ln ln n = {L/math.log(L):.2f},  bound {math.ceil(3*L/math.log(L))}')

The maximum load grows, but extremely slowly, and stays between $\frac{\ln n}{\ln\ln n}$ and the bound $\frac{3\ln n}{\ln\ln n}$; the true asymptotics is $\frac{\ln n}{\ln\ln n}(1+o(1))$ (Gonnet, 1981), with a slowly vanishing $o(1)$. Now two choices:

In [ ]:
def two_choices(n, d=2):
    load = np.zeros(n, dtype=np.int64)
    choices = rng.integers(0, n, (n, d))
    for c in choices:
        j = c[np.argmin(load[c])]
        load[j] += 1
    return int(load.max())
for e in range(3, 7):
    n = 10**e
    print(f'n = 10^{e}: two choices max load {two_choices(n)},  ln ln n / ln 2 = {math.log(math.log(n))/math.log(2):.2f}')

With two choices the maximum load drops from about $\frac{\ln n}{\ln\ln n}$ to $\frac{\ln\ln n}{\ln2}+O(1)$, an exponential improvement (Azar, Broder, Karlin and Upfal, 1994). Load balancers and hash tables (cuckoo hashing) use this *power of two choices*.

**P4.4.** We derive the $k$ positions from one SHA-256 digest by double hashing, $h_1+ih_2 \bmod m$, a standard trick (Kirsch and Mitzenmacher, 2006).

In [ ]:
class Bloom:
    def __init__(self, m, k):
        self.m, self.k, self.bits = m, k, np.zeros(m, dtype=bool)
    def positions(self, key):
        d = hashlib.sha256(key.encode()).digest()
        h1, h2 = int.from_bytes(d[:8], 'little'), int.from_bytes(d[8:16], 'little') | 1
        return [(h1 + i * h2) % self.m for i in range(self.k)]
    def add(self, key):
        self.bits[self.positions(key)] = True
    def __contains__(self, key):
        return all(self.bits[self.positions(key)])
n = 20_000; m = 10 * n
members = [f'user{i}' for i in range(n)]
others = [f'other{i}' for i in range(200_000)]
for k in range(1, 15):
    bf = Bloom(m, k)
    for w in members: bf.add(w)
    assert all(w in bf for w in members[:1000])   # no false negatives
    fp = sum(w in bf for w in others) / len(others)
    print(f'k = {k:2d}: zero bits {1 - bf.bits.mean():.4f} (theory {math.exp(-k*n/m):.4f}),  '
          f'false positives {fp:.5f} (theory {(1 - math.exp(-k*n/m))**k:.5f})')
print('optimum k = (m/n) ln 2 =', 10 * math.log(2), ' rate 0.6185^(m/n) =', 0.6185**10)

The measured rates follow the formula closely, with the minimum at $k=7$ and about $0.8\%$ false positives, using $10$ bits per key whatever the length of the keys. A Bloom filter never gives a false negative: a key that was inserted always has its $k$ bits set.

**P4.5.** One estimate is $4\times$(fraction of points in the quarter disc).

In [ ]:
N = 400
def estimates(size):
    return 4 * (rng.random((size, N))**2 + rng.random((size, N))**2 <= 1).mean(axis=1)
one = estimates(200_000)
eps = np.quantile(np.abs(one - math.pi), 0.75)
print(f'eps = {eps:.4f}:  P(one estimate within eps) = {np.mean(np.abs(one - math.pi) <= eps):.3f}')
for k in [1, 3, 5, 10, 20]:
    runs = estimates(20_000 * (2 * k + 1)).reshape(20_000, 2 * k + 1)
    med_fail = np.mean(np.abs(np.median(runs, axis=1) - math.pi) > eps)
    mean_fail = np.mean(np.abs(runs.mean(axis=1) - math.pi) > eps)
    print(f'2k+1 = {2*k+1:2d}: median fails {med_fail:.5f}, mean fails {mean_fail:.5f}, bound e^(-(2k+1)/8) = {math.exp(-(2*k+1)/8):.4f}')

The median's failure rate falls exponentially, much faster than the bound (which only uses $P(\text{bad})\le\frac14$). Here the *mean* does even better, since averaging $2k+1$ estimates is the same as one estimate with $(2k+1)N$ points, and its error shrinks like $\frac1{\sqrt{2k+1}}$. The median trick matters when the answers can be wildly wrong with probability up to $\frac14$, for instance in streaming algorithms that estimate the number of distinct elements: one absurd answer spoils a mean, but not a median.

**P4.6.** Huffman's algorithm with a heap: repeatedly merge the two lightest subtrees; each merge adds one bit to every symbol below it.

In [ ]:
def huffman_lengths(probs):
    # probs: dict symbol -> probability; returns dict symbol -> codeword length
    if len(probs) == 1:
        return {s: 1 for s in probs}
    heap = [(p, i, [s]) for i, (s, p) in enumerate(probs.items())]
    heapq.heapify(heap)
    length = {s: 0 for s in probs}
    counter = len(heap)
    while len(heap) > 1:
        p1, _, s1 = heapq.heappop(heap)
        p2, _, s2 = heapq.heappop(heap)
        for s in s1 + s2:
            length[s] += 1
        heapq.heappush(heap, (p1 + p2, counter, s1 + s2)); counter += 1
    return length
def entropy(probs):
    return -sum(p * log2(p) for p in probs.values() if p > 0)
p = 0.1
H = entropy({0: 1 - p, 1: p})
for n in range(1, 11):
    blocks = {}
    for b in range(2**n):
        ones = bin(b).count('1')
        blocks[b] = p**ones * (1 - p)**(n - ones)
    lens = huffman_lengths(blocks)
    Ln = sum(blocks[b] * lens[b] for b in blocks)
    assert sum(2.0**-l for l in lens.values()) <= 1 + 1e-12      # Kraft
    assert n * H - 1e-12 <= Ln < n * H + 1
    print(f'n = {n:2d}: L_n/n = {Ln/n:.4f}   (H = {H:.4f}, H + 1/n = {H + 1/n:.4f})')

The rate per bit decreases towards $H=h(0.1)\approx0.469$, as Theorem 4.19(c) promises; the bound $H+\frac1n$ is not tight, and Huffman does much better in practice. Now English, with the closing paragraph of Darwin's *Origin of Species* (1859, public domain).

In [ ]:
text = '''It is interesting to contemplate an entangled bank, clothed with many plants of many kinds, with birds
singing on the bushes, with various insects flitting about, and with worms crawling through the damp earth,
and to reflect that these elaborately constructed forms, so different from each other, and dependent on each
other in so complex a manner, have all been produced by laws acting around us. These laws, taken in the largest
sense, being Growth with Reproduction; Inheritance which is almost implied by reproduction; Variability from
the indirect and direct action of the external conditions of life, and from use and disuse; a Ratio of Increase
so high as to lead to a Struggle for Life, and as a consequence to Natural Selection, entailing Divergence of
Character and the Extinction of less-improved forms. Thus, from the war of nature, from famine and death, the
most exalted object which we are capable of conceiving, namely, the production of the higher animals, directly
follows. There is grandeur in this view of life, with its several powers, having been originally breathed into a
few forms or into one; and that, whilst this planet has gone cycling on according to the fixed law of gravity,
from so simple a beginning endless forms most beautiful and most wonderful have been, and are being, evolved.'''
s = ''.join(c if c.isalpha() else ' ' for c in text.lower())
s = ' '.join(s.split())                      # 27 symbols: a-z and space
N = len(s)
freq = {c: v / N for c, v in Counter(s).items()}
lens = huffman_lengths(freq)
print(f'{N} characters, {len(freq)} distinct symbols, log2(27) = {log2(27):.3f}')
print(f'letter entropy H = {entropy(freq):.3f} bits,  Huffman on letters: {sum(freq[c]*lens[c] for c in freq):.3f} bits/char')
pairs = [s[i:i+2] for i in range(0, N - 1, 2)]
pf = {c: v / len(pairs) for c, v in Counter(pairs).items()}
pl = huffman_lengths(pf)
print(f'pair entropy / 2 = {entropy(pf)/2:.3f},  Huffman on pairs: {sum(pf[c]*pl[c] for c in pf)/2:.3f} bits/char')
print(f'zlib: {8*len(zlib.compress(s.encode(), 9))/N:.3f} bits/char')

Letters alone carry about $4.1$ bits each, below the $\log_227\approx4.75$ of a uniform alphabet, and Huffman comes within a few hundredths of a bit of it. Pairs do better, because English letters are far from independent; but on a text this short the pair statistics are over-fitted (most pairs occur once or twice), so this number is optimistic: the code table itself would cost more than it saves. `zlib`, which pays for everything it uses, gets about $3.9$ bits per character: it exploits repeated words, and it wins clearly on long texts. Shannon estimated, with human guessing experiments, that English carries about $1$ bit per letter once all the context is used, which is roughly what modern language-model compressors achieve.

**P4.7.** The probability of a sequence depends only on its number $k$ of ones, so we sum over $k$ with the multiplicity $\binom nk$.

In [ ]:
p, eps = 0.2, 0.05
H = entropy({0: 1 - p, 1: p})
for n in [100, 500, 2000, 10_000]:
    size, prob = 0, 0.0
    for k in range(n + 1):
        v = -(k * log2(p) + (n - k) * log2(1 - p)) / n       # -(1/n) log2 p(x)
        if abs(v - H) <= eps:
            size += comb(n, k)
            prob += math.exp(math.lgamma(n+1) - math.lgamma(k+1) - math.lgamma(n-k+1) + k*log(p) + (n-k)*log(1-p))
    print(f'n = {n:6d}: P(T) = {prob:.4f},  log2|T|/n = {math.log2(size)/n:.4f}  in [H-eps, H+eps] = [{H-eps:.3f}, {H+eps:.3f}]')

The typical set carries more and more of the probability, while its size $2^{n(H+\varepsilon')}$ has an exponent that settles near $H+\varepsilon$: within the band, most sequences sit at the edge with the most ones, because the number of sequences with $k$ ones grows with $k$ up to $\frac n2$. This is the usual tension in the AEP: the typical set is small compared with all $2^n$ sequences, but its size is only pinned down to within $2^{\pm n\varepsilon}$.

**P4.8.** Run lengths with numpy: find where the sequence changes value.

In [ ]:
def longest_heads(a):
    # a: 0/1 array; longest run of ones
    d = np.diff(np.concatenate(([0], a, [0])))
    starts, ends = np.flatnonzero(d == 1), np.flatnonzero(d == -1)
    return int((ends - starts).max()) if len(starts) else 0
def longest_any(a):
    d = np.flatnonzero(np.diff(a) != 0)
    edges = np.concatenate(([-1], d, [len(a) - 1]))
    return int(np.diff(edges).max())
fair = rng.integers(0, 2, (20_000, 200))
LH = np.array([longest_heads(a) for a in fair]); LA = np.array([longest_any(a) for a in fair])
print('fair coin, 200 tosses: mean longest run of heads', LH.mean(), ' of either side', LA.mean())
print('   P(some run >= 6)', np.mean(LA >= 6), '  distribution of longest heads run:', np.bincount(LH)[:14] / len(LH))
def faker(n):
    out = [rng.integers(0, 2)]
    for _ in range(n - 1):
        out.append(1 - out[-1] if rng.random() < 0.6 else out[-1])
    return np.array(out)
fake = [longest_any(faker(200)) for _ in range(5000)]
print('faker (switch prob 0.6): mean longest run', np.mean(fake), '  P(some run >= 6)', np.mean(np.array(fake) >= 6))
big = rng.integers(0, 2, 10**7)
for e in range(2, 8):
    n = 10**e
    print(f'n = 10^{e}: L_n = {longest_heads(big[:n])},  L_n / log2 n = {longest_heads(big[:n]) / math.log2(n):.3f}')

A fair coin produces a run of $6$ identical outcomes in $200$ tosses with probability $97\%$; the faker, by alternating a little too eagerly, does so only about $70\%$ of the time and has shorter longest runs on average, which is how a teacher can spot the invented sequences. Along one long sequence, $\frac{L_n}{\log_2n}$ creeps towards $1$; the convergence is slow, since $L_n-\log_2n$ stays of order $1$ and fluctuates without converging (its law is close to a discretised Gumbel law).